# EDA: Departures by country of birth and sex

**Data source:** DuckDB view `departures` (loaded from `data/processed/departures.parquet`)  
**Run `notebooks/load/03_load_departures.ipynb` first.**

**Charts produced:**
- `departures_aruba_neth_ant_by_sex.png` — Aruba/Neth. Ant. departures by sex, 2015–2023
- `departures_trends_by_country_sex.png` — subplot grid: all countries, departures by sex

---
## 1. Setup

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent if Path.cwd().name == "eda" else Path.cwd()))

from config.project_paths import DB_FILES, FIGURES

In [ ]:
con = duckdb.connect(str(DB_FILES / "aruba.duckdb"))

# Quick check
print(con.execute("SELECT COUNT(*) FROM departures").fetchone())
print(con.execute("SELECT DISTINCT country FROM departures ORDER BY country").df())

---
## 2. Load data from DuckDB

Exclude the total row for country-level charts.

In [ ]:
df = con.execute("""
    SELECT country, year, sex, value
    FROM departures
    WHERE country != 'Total Departures:'
    ORDER BY country, year, sex
""").df()

con.close()

print(f"Shape: {df.shape}")
df.head()

---
## 3. Chart 1: Aruba / Neth. Ant. departures by sex

In [ ]:
df_aruba = df[df["country"] == "Aruba/ Neth. Ant."].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.grid(alpha=0.25)

for sex, group in df_aruba.groupby("sex"):
    ax.plot(group["year"], group["value"], marker="o", label=sex)

ax.set_title("Aruba / Neth. Ant. — departures by sex, 2015–2023")
ax.set_xlabel("Year")
ax.set_ylabel("Departures")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "departures_aruba_neth_ant_by_sex.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", FIGURES / "departures_aruba_neth_ant_by_sex.png")

---
## 4. Chart 2: All countries — subplot grid by sex

In [ ]:
countries = sorted(df["country"].unique())

ncols = 2
nrows = (len(countries) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(14, 4 * nrows),
    sharex=True, sharey=True
)
axes = axes.flatten()

for ax, country in zip(axes, countries):
    country_data = df[df["country"] == country]
    for sex, group in country_data.groupby("sex"):
        ax.plot(group["year"], group["value"], marker="o", label=sex)
    ax.set_title(country)
    ax.set_xlabel("Year")
    ax.set_ylabel("Departures")
    ax.grid(True, alpha=0.25)

# Remove unused axes
for ax in axes[len(countries):]:
    fig.delaxes(ax)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=1)
fig.suptitle("Departure trends by country and sex, 2015–2023", y=1.01, fontsize=13)
fig.tight_layout()

fig.savefig(FIGURES / "departures_trends_by_country_sex.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", FIGURES / "departures_trends_by_country_sex.png")

---
## 5. Optional: Net migration snapshot

Cross-reference domiciliation vs departures if both views are loaded.
Uncomment when `domiciliation` view is also available.

In [ ]:
# con = duckdb.connect(str(DB_FILES / "aruba.duckdb"))
#
# net_df = con.execute("""
#     SELECT
#         d.country,
#         d.year,
#         d.sex,
#         dom.value AS domiciliations,
#         d.value   AS departures,
#         dom.value - d.value AS net_migration
#     FROM departures d
#     JOIN domiciliation dom
#         ON d.country = dom.country
#         AND d.year = dom.year
#         AND d.sex = dom.sex
#     WHERE d.country != 'Total Departures:'
#       AND dom.country != 'Total Domiciliation:'
#     ORDER BY d.country, d.year, d.sex
# """).df()
#
# con.close()
# net_df.head(12)